# Iceberg: offline package-based rerun

Attach the code-wheel dataset named in the metadata and accept the competition rules.
This is a bounded method-port rerun, not a reproduction of the historical leaderboard score.
It trains the package's original-shape CNN on 128 training records, evaluates the full saved
holdout, and predicts the full test input. Internet and private services are not required.

In [ ]:
from pathlib import Path
import subprocess
import sys

wheel_roots = [p for p in Path("/kaggle/input").rglob("iceberg_classifier-*.whl")]
assert len(wheel_roots) == 1, "Attach exactly one project wheel"
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                str(wheel_roots[0])], check=True)


In [ ]:
from iceberg.runtime import train, predict
import json
import zipfile

competition = "statoil-iceberg-classifier-challenge"
candidates = [Path("/kaggle/input/competitions") / competition,
              Path("/kaggle/input") / competition]
data_dir = next(p for p in candidates if (p / "train.json").is_file())
run_dir = Path("/kaggle/working/iceberg-run")
config = {'name': 'bounded-real-data', 'seed': 1017, 'split_seed': 25, 'validation_fraction': 0.159, 'filters': 8, 'hidden': 128, 'batch_size': 16, 'epochs': 1, 'epoch_repeats': 1, 'learning_rate': 0.0005, 'learning_rate_decay': 0.002, 'augment': True, 'max_train_rows': 128, 'threads': 4}
manifest = train(data_dir / "train.json", run_dir, config)
receipt = predict(run_dir / "model.pt", data_dir / "test.json",
                  Path("/kaggle/working/submission.csv"))
with zipfile.ZipFile("/kaggle/working/run-bundle.zip", "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in run_dir.iterdir():
        if path.is_file():
            bundle.write(path, path.name)
print(json.dumps({"run_id": manifest["logical_run_id"], "prediction": receipt}, indent=2))
